# 🌸 Classificando Flores com Redes Neurais

Este notebook é um guia **bem passo a passo** para quem está vendo PyTorch pela primeira vez. Vamos usar o famoso **dataset Iris** para ensinar uma rede neural a reconhecer o tipo de uma flor a partir de 4 medidas simples: comprimento e largura da sépala e da pétala.

## 0. Preparando o ambiente

Antes de tudo, precisamos importar as bibliotecas que vamos usar:

- **pandas**: para carregar e explorar os dados em formato de tabela;
- **seaborn / matplotlib**: para fazer gráficos;
- **scikit-learn (sklearn)**: para preparar os dados (normalizar, dividir treino/teste, codificar rótulos) e calcular métricas;
- **torch (PyTorch)**: a biblioteca que vamos usar para construir e treinar a rede neural.

Se alguma dessas bibliotecas não estiver instalada, rode no terminal: `pip install torch pandas scikit-learn seaborn matplotlib`.

In [ ]:
import torch
import torch.nn as nn                              # módulo para definir a rede neural
from torch.utils.data import Dataset, DataLoader   # utilidades para organizar os dados

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

# Fixamos uma "semente" aleatória para que os resultados sejam sempre os mesmos quando você rodar de novo
torch.manual_seed(42)

## 1. Conhecendo o dataset Iris

O dataset Iris é o "Hello World" do aprendizado de máquina. Ele tem **150 flores**, cada uma descrita por 4 medidas (em centímetros):

- `SepalLengthCm` — comprimento da sépala
- `SepalWidthCm` — largura da sépala
- `PetalLengthCm` — comprimento da pétala
- `PetalWidthCm` — largura da pétala

E uma coluna `Species` que diz a qual das 3 espécies a flor pertence: *Iris-setosa*, *Iris-versicolor* ou *Iris-virginica*.

<img src="https://www.embedded-robotics.com/wp-content/uploads/2022/01/Iris-Dataset-Classification.png" width="500"/>

*Imagem: https://www.embedded-robotics.com/iris-dataset-classification/*

**Nosso objetivo**: dado o tamanho de sépala e pétala de uma flor nova, a rede neural deve identificar a espécie correta. Isso é um problema de **classificação** (escolher entre categorias), diferente de **regressão** (prever um número contínuo).

In [ ]:
# Vamos carregar os dados direto do repositório público do curso
df = pd.read_csv('https://raw.githubusercontent.com/mdrs-thiago/PUC_Redes_Neurais/main/datasets/iris.csv')

# .head() mostra as 5 primeiras linhas, só para conferirmos que carregou certo
df.head()

### Dando uma olhada geral nos dados

Antes de sair treinando redes neurais, é sempre bom **entender os dados**. Algumas perguntas úteis:

- Quantas flores de cada espécie temos?
- As medidas têm uma escala parecida entre si?
- Existe alguma diferença visual clara entre as espécies?

In [ ]:
# describe() calcula estatísticas básicas (média, desvio padrão, mínimo, máximo...) de cada coluna numérica
df.describe()

In [ ]:
# Quantas flores existem de cada espécie?
df['Species'].value_counts()

In [ ]:
# groupby + mean() mostra a média de cada medida, separada por espécie.
# Repare como a pétala (Petal) muda bastante de tamanho entre espécies — isso vai ajudar bastante a rede a diferenciá-las!
df.groupby(by='Species').mean(numeric_only=True)

In [ ]:
# pairplot desenha um gráfico de dispersão para cada par de colunas, colorindo por espécie.
# Isso nos dá uma boa intuição visual de que as espécies são, em geral, separáveis.
sns.pairplot(df.drop(columns=['Id']), hue='Species')
plt.show()

## 2. Preparando os dados para a rede neural

Uma rede neural não entende texto ("Iris-setosa") e possui umna melhor qualidade de resultado quando usamos números pequenos. Precisamos fazer 3 atividades antes de treinar:

1. **Separar entradas (X) e saídas (y)** — o que a rede vai receber e o que ela deve prever;
2. **Dividir em treino, validação e teste** — para conseguirmos avaliar se a rede realmente aprendeu, e não apenas "decorou" os exemplos;
3. **Normalizar as entradas e codificar os rótulos** — colocar tudo em uma escala numérica amigável para a rede.

### 2.1 Separando X (entradas) e y (saída)

- `X` são as colunas que a rede vai usar para decidir: as 4 medidas da flor.
- `y` é o que queremos prever: a espécie.

A coluna `Id` é descartada porque é só um número de identificação, não tem relação com a espécie da flor.

In [ ]:
y = df['Species'].values
X = df.drop(columns=['Id', 'Species']).values

print('Formato de X (linhas, colunas):', X.shape)
print('Formato de y (linhas,):', y.shape)

### 2.2 Dividindo em treino, validação e teste

Além do **treino** (para a rede aprender) e do **teste** (para avaliar de forma justa no final, com dados que a rede nunca viu), vamos separar também um conjunto de **validação**.

**Por que validação?** O conjunto de teste só deve ser usado **uma vez**, ao final de tudo, para simular "o mundo real". Se ficássemos olhando o desempenho no teste a cada época de treino para decidir se a rede está indo bem, estaríamos indiretamente "vazando" informação do teste para as nossas decisões (por exemplo, quantas épocas treinar). O conjunto de validação resolve isso: é um pedaço dos dados que a rede também não usa para aprender, mas que podemos consultar durante o treino, quantas vezes quisermos, para acompanhar se ela está generalizando bem ou apenas decorando o treino (*overfitting*).

Vamos usar a proporção **60% treino / 20% validação / 20% teste**, feita em duas etapas:

1. Primeiro separamos 20% para teste;
2. Depois, dos 80% restantes, separamos 25% para validação (25% de 80% = 20% do total).

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% dos dados ficam de fora para teste
    random_state=42,    # garante que a divisão seja sempre a mesma ao reexecutar
    stratify=y          # mantém a proporção de cada espécie igual em treino e teste
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,     # 25% dos 80% restantes = 20% do total
    random_state=42,
    stratify=y_train_val
)

print('Exemplos de treino    :', len(X_train))
print('Exemplos de validação :', len(X_val))
print('Exemplos de teste     :', len(X_test))

### 2.3 Normalizando as medidas (`MinMaxScaler`)

As redes neurais aprendem melhor quando todas as entradas estão em escalas parecidas. Aqui, a largura da pétala varia entre ~0.1 e ~2.5, enquanto o comprimento da sépala varia entre ~4.3 e ~7.9 — escalas diferentes podem confundir o treinamento.

O `MinMaxScaler` transforma cada coluna para o intervalo `[0, 1]`, usando a fórmula:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

⚠️ **Detalhe importante**: aprendemos (`fit`) o mínimo e o máximo **apenas com os dados de treino**, e aplicamos (`transform`) essa mesma regra na validação e no teste. Se usássemos validação ou teste para calcular o mínimo/máximo, estaríamos dando uma vantagem injusta ao modelo (isso se chama *data leakage*, ou vazamento de dados).

In [ ]:
scaler = MinMaxScaler()

X_train_std = scaler.fit_transform(X_train)   # aprende min/max no treino e já transforma
X_val_std = scaler.transform(X_val)            # aplica a mesma transformação na validação
X_test_std = scaler.transform(X_test)          # aplica a mesma transformação no teste

### 2.4 Convertendo os rótulos de texto para número (`LabelEncoder`)

A rede neural não entende a palavra "Iris-setosa", ela precisa de números. O `LabelEncoder` faz essa tradução, por exemplo:

- `Iris-setosa` → `0`
- `Iris-versicolor` → `1`
- `Iris-virginica` → `2`

In [ ]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train)
y_val_encoded = encoder.transform(y_val)
y_test_encoded = encoder.transform(y_test)

# encoder.classes_ guarda a ordem usada na tradução
list(enumerate(encoder.classes_))

## 3. Tensores

Um **tensor** é, na prática, muito parecido com um array do NumPy — uma lista (ou tabela, ou cubo) de números. A diferença é que tensores do PyTorch sabem como calcular **gradientes** automaticamente (usaremos isso no treinamento) e podem rodar em GPU com maior facilidade.

Vamos brincar um pouco com tensores antes de seguir:

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([0.0, 1.0, 0.0])

print('a * b       =', a * b)          # multiplicação elemento a elemento
print('soma de a   =', torch.sum(a))   # soma todos os elementos
print('máximo de a =', torch.max(a))   # maior elemento

Nossos dados hoje estão como arrays do NumPy (vindos do scikit-learn). Vamos precisar transformá-los em tensores antes de entregá-los à rede — faremos isso já dentro do `Dataset` customizado, na próxima seção.

## 4. Organizando os dados: `Dataset` e `DataLoader`

Pense num `Dataset` como organizador para leitura de dados e submissão dos mesmos para a rede. O PyTorch exige que a gente implemente duas coisas:

- `__len__`: quantos exemplos existem no total;
- `__getitem__`: como pegar o exemplo de número `index` (a entrada `X` e a saída `y` correspondente).

Já o `DataLoader` é quem **entrega os dados batches** durante o treino, podendo embaralhar a ordem a cada época. 

In [ ]:
class IrisDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data

    def __getitem__(self, index):
        # converte a linha de entrada em tensor de números decimais (float)
        X_batch = torch.from_numpy(self.X_data[index]).float()
        # converte o rótulo em tensor de número inteiro (long), exigido pela função de perda que usaremos
        y_batch = torch.tensor(self.y_data[index]).long()
        return X_batch, y_batch

    def __len__(self):
        return len(self.X_data)


train_dataset = IrisDataset(X_train_std, y_train_encoded)
val_dataset = IrisDataset(X_val_std, y_val_encoded)
test_dataset = IrisDataset(X_test_std, y_test_encoded)

# Vamos espiar um único exemplo para ver o formato
X_exemplo, y_exemplo = train_dataset[0]
print('Entrada (medidas normalizadas):', X_exemplo)
print('Saída (espécie codificada)    :', y_exemplo)

In [ ]:
# batch_size=16 significa que a rede vai olhar 16 flores de cada vez antes de atualizar seus pesos
# shuffle=True embaralha a ordem dos exemplos a cada época, o que ajuda a rede a não "decorar a ordem"
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

## 5. Montando a rede neural

Uma rede neural é feita de **camadas** empilhadas. Cada camada `nn.Linear(n_in, n_out)` faz uma combinação linear das entradas e entre elas colocamos **funções de ativação**, que introduzem não-linearidade — sem elas, empilhar camadas lineares seria matematicamente equivalente a ter uma única camada.

Nossa arquitetura vai ser simples:

```
entrada (4 medidas) → camada oculta (com ativação ReLU) → saída (3 espécies)
```

No PyTorch, criamos uma rede definindo uma classe que herda de `nn.Module`, com dois métodos:

- `__init__`: aqui declaramos quais camadas a rede vai ter;
- `forward`: aqui descrevemos a **ordem** em que os dados passam por essas camadas.

In [ ]:
class IrisClassifier(nn.Module):
    def __init__(self, n_in=4, n_out=3, hidden_size=16):
        super().__init__()

        self.fc1 = nn.Linear(in_features=n_in, out_features=hidden_size)  # camada de entrada -> oculta
        self.activation = nn.ReLU()                                       # não-linearidade
        self.fc2 = nn.Linear(in_features=hidden_size, out_features=n_out)  # camada oculta -> saída

    def forward(self, x):
        h1 = self.fc1(x)          # combinação linear das 4 medidas
        a1 = self.activation(h1)  # aplica a não-linearidade
        out = self.fc2(a1)        # gera 3 números (um "placar" para cada espécie)
        return out


model = IrisClassifier(n_in=4, n_out=3, hidden_size=16)
model

## 6. Função de perda e otimizador

Para treinar, a rede precisa de duas coisas:

- **Função de perda (loss)**: mede o quão distante está a previsão da rede em relação à resposta certa. Para classificação com várias classes, usamos a `CrossEntropyLoss` — ela já cuida de transformar os logits em probabilidades internamente.
- **Otimizador**: o algoritmo que ajusta os pesos da rede para diminuir a perda a cada passo. Vamos usar o `Adam`, um dos otimizadores mais populares e que costuma funcionar bem com os parâmetros padrões.


In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # lr = learning rate: o "tamanho do passo" de cada ajuste

## 7. Treinando a rede

Uma **época (epoch)** é uma passada completa por todos os exemplos de treino. Em cada época, para cada lote (`batch`) de flores:

1. Zeramos os gradientes acumulados (`optimizer.zero_grad()`);
2. Passamos as entradas pela rede para obter as previsões (`model(X_batch)`);
3. Calculamos o quão erradas estão essas previsões (`loss_function(...)`);
4. Calculamos os gradientes — o quanto cada peso contribuiu para o erro (`loss.backward()`);
5. Damos um passo do otimizador, ajustando os pesos na direção que reduz o erro (`optimizer.step()`).

Isso se chama **retropropagação (backpropagation)**, e o PyTorch calcula tudo automaticamente através do recurso chamado `autograd` — você não precisa fazer nenhuma conta de derivada na mão!

Ao final de cada época, também vamos calcular a perda no conjunto de **validação** (sem treinar com ele — só olhando o quão bem a rede está indo em dados que ela não usou para aprender). Isso nos ajuda a perceber, por exemplo, se a rede começa a "decorar" o treino (*overfitting*): quando isso acontece, a perda de treino continua caindo, mas a perda de validação para de cair ou até aumenta.

In [ ]:
epochs = 100
historico_perda_treino = []
historico_perda_val = []

for epoch in range(epochs):
    # --- Treino ---
    model.train()  # coloca o modelo em modo de treino
    perda_epoca = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # 1. limpa gradientes antigos

        y_hat = model(X_batch)                # 2. previsão da rede
        loss = loss_function(y_hat, y_batch)  # 3. calcula o erro

        loss.backward()                       # 4. calcula os gradientes
        optimizer.step()                      # 5. atualiza os pesos

        perda_epoca += loss.item()

    historico_perda_treino.append(perda_epoca / len(train_loader))

    # --- Validação ---
    model.eval()  # modo de avaliação: não estamos treinando com esses dados
    perda_val_epoca = 0.0

    with torch.no_grad():  # não precisamos calcular gradientes aqui
        for X_batch, y_batch in val_loader:
            y_hat = model(X_batch)
            loss = loss_function(y_hat, y_batch)
            perda_val_epoca += loss.item()

    historico_perda_val.append(perda_val_epoca / len(val_loader))

    if (epoch + 1) % 10 == 0:
        print(f'Época [{epoch+1}/{epochs}] - Loss treino: {historico_perda_treino[-1]:.4f} '
              f'- Loss validação: {historico_perda_val[-1]:.4f}')

### Visualizando o aprendizado

Se tudo deu certo, a perda de treino deve **cair ao longo das épocas** — sinal de que a rede está errando cada vez menos. Vamos plotar junto a perda de validação: se as duas curvas caem juntas e ficam próximas, a rede está generalizando bem; se a curva de validação parar de cair (ou subir) enquanto a de treino continua caindo, é sinal de *overfitting*.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico_perda_treino, label='Treino')
plt.plot(historico_perda_val, label='Validação')
plt.title('Loss ao longo do treinamento')
plt.xlabel('Época')
plt.ylabel('Loss (CrossEntropyLoss)')
plt.legend()
plt.grid(True)
plt.show()

## 8. Avaliando o modelo
Vamos testar a rede em flores que ela **nunca viu** durante o treino (o conjunto de teste).


Dois detalhes importantes:

- `model.eval()`: avisa o PyTorch que estamos avaliando, não treinando (importante para camadas como `Dropout`/`BatchNorm`, que se comportam diferente em treino e teste — não usamos aqui, mas é uma boa prática sempre incluir);
- `torch.no_grad()`: desliga o cálculo de gradientes, já que não vamos treinar aqui — isso deixa a avaliação mais rápida e economiza memória.

In [ ]:
model.eval()

with torch.no_grad():
    X_test_tensor = torch.from_numpy(X_test_std).float()
    y_hat = model(X_test_tensor)                      # "placares" para cada uma das 3 espécies
    y_pred = torch.argmax(y_hat, dim=1).numpy()        # escolhe a espécie com maior placar

### Métricas de avaliação

- **Precisão (precision)**: das vezes que a rede disse "é a espécie X", quantas vezes ela acertou?
- **Revocação (recall)**: das flores que realmente eram da espécie X, quantas a rede conseguiu identificar?
- **F1-score**: uma média entre precisão e recall;
- **Matriz de confusão**: uma tabela mostrando, para cada espécie real, quantas vezes a rede previu cada espécie (a diagonal são os acertos).

In [ ]:
print(classification_report(y_test_encoded, y_pred, target_names=encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test_encoded, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel('Espécie prevista')
plt.ylabel('Espécie real')
plt.title('Matriz de confusão')
plt.show()

## 9. Testando com uma flor nova

Por fim, vamos simular o uso real do modelo: alguém mede uma flor nova e quer saber a espécie. Repare que precisamos aplicar **o mesmo `scaler`** usado no treino, para manter a mesma escala de valores.

Aqui também vamos aplicar `torch.softmax` sobre os logits só para **mostrar as probabilidades** de cada espécie — como discutimos na seção 5, isso é feito manualmente na hora de interpretar a saída, e não como uma camada da rede.

In [ ]:
import numpy as np

# Exemplo: sépala 5.9cm x 3.0cm, pétala 5.1cm x 1.8cm
flor_nova = np.array([[5.9, 3.0, 5.1, 1.8]])
flor_nova_std = scaler.transform(flor_nova)

model.eval()
with torch.no_grad():
    entrada = torch.from_numpy(flor_nova_std).float()
    logits = model(entrada)                              # "placares" crus (sem softmax)
    probabilidades = torch.softmax(logits, dim=1)         # convertemos para probabilidades só aqui, na hora de interpretar
    especie_prevista = torch.argmax(probabilidades, dim=1).item()

for especie, prob in zip(encoder.classes_, probabilidades[0]):
    print(f'{especie:20s}: {prob.item()*100:5.1f}%')

print('\nEspécie prevista:', encoder.classes_[especie_prevista])